In [3]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.tools import TavilySearchResults

# 🔐 Load API keys
load_dotenv(".env")
#openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# 🔸 Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# ✅ Tool 1: Simple QA Tool
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
qa_tool = Tool(
    name="Simple QA",
    func=qa_chain.run,
    description="Answers factual questions clearly"
)

# ✅ Tool 2: Summarizer Tool
summary_prompt = PromptTemplate.from_template("Summarize this text:\n\n{text}")
summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
summary_tool = Tool(
    name="Summarizer",
    func=summary_chain.run,
    description="Summarizes long paragraphs or text content"
)

# ✅ Tool 3: Web Search Tool (Tavily)
search_tool = Tool(
    name="Web Search",
    func=TavilySearchResults(max_results=3).run,
    description="Search the internet for current and live information"
)

# 🔧 Wrap all tools in an agent
tools = [qa_tool, summary_tool, search_tool]

agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,  # shows reasoning process
    handle_parsing_errors=True

)

# 🚀 Run user queries
queries = [
    "What is LangGraph in LangChain?"
    #"Summarize this: LangChain is a framework to build LLM apps using prompts, memory, tools, and agents.",
    #"Latest news about OpenAI GPT-5.6"
]

for query in queries:
    print("\n🧑‍💻 User Query:", query)
    response = agent_executor.run(query)
    print("\n🤖 Agent Response:", response)



🧑‍💻 User Query: What is LangGraph in LangChain?


> Entering new AgentExecutor chain...
Action: Web Search
Action Input: What is LangGraph in LangChain?
Observation: [{'title': 'What is LangGraph?', 'url': 'https://www.ibm.com/think/topics/langgraph', 'content': '# What is LangGraph?\n\n## LangGraph overview\n\nLangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. [...] LangGraph is also built on several key technologies, including LangChain, a Python framework for building AI applications. LangChain includes a library for building and managing LLMs. LangGraph also uses the human-i

In [4]:
response

'LangGraph, created by the LangChain team, is an open-source AI agent framework designed to build, deploy, and manage complex generative AI agent workflows. It distinguishes itself by organizing tasks into a graph structure, where each "node" represents a task and "edges" define how these tasks connect. This graph-based approach enables the creation of workflows that can branch, loop, and maintain state, allowing agents to handle more complex, adaptive, and stateful scenarios than traditional linear chains.\n\nWhile LangChain is a broader framework for designing LLM-powered AI applications, offering abstractions and integrations for models, tools, and agent loops, LangGraph serves as a lower-level orchestration runtime. Its primary focus is on providing core capabilities essential for agent orchestration, such as durable execution, streaming, human-in-the-loop interactions, and persistence for long-running, stateful agents. Although it can leverage LangChain components, LangGraph is de